In [1]:
print("Streamlit setup cell ✅")

Streamlit setup cell ✅


## 2. Imports used by the notebook

In [2]:
from pathlib import Path

print("Notebook ready! ✅")


Notebook ready! ✅


In [3]:
app_code = 'import requests\nimport streamlit as st\nimport pandas as pd\nimport folium\nfrom streamlit_folium import st_folium\n\n\nAPI_BASE_URL = "http://127.0.0.1:8000"\n\nst.set_page_config(\n    page_title="TravelMate AI",\n    page_icon="🌍",\n    layout="wide"\n)\n\nst.title("🌍 TravelMate AI")\nst.subheader("Your personalized AI travel planner ✈️")\n\nst.write(\n    "Tell TravelMate what kind of trip you want, "\n    "and it will recommend places and build a day-by-day itinerary."\n)\n\n# -------------------------------------------------\n# Sidebar: Trip preferences\n# -------------------------------------------------\n\nst.sidebar.header("🧳 Trip Preferences")\n\ndestination = st.sidebar.text_input(\n    "Destination",\n    value="Manali"\n)\n\ndays = st.sidebar.number_input(\n    "Number of days",\n    min_value=1,\n    max_value=14,\n    value=3,\n    step=1\n)\n\ntop_n = st.sidebar.slider(\n    "Number of recommendations",\n    min_value=3,\n    max_value=15,\n    value=6\n)\n\nst.sidebar.markdown("### ❤️ What do you like?")\n\nnature = st.sidebar.slider(\n    "🌿 Nature",\n    0.0, 1.0, 0.8, 0.1\n)\n\nhistory = st.sidebar.slider(\n    "🏛️ History",\n    0.0, 1.0, 0.2, 0.1\n)\n\nculture = st.sidebar.slider(\n    "🎭 Culture",\n    0.0, 1.0, 0.3, 0.1\n)\n\nadventure = st.sidebar.slider(\n    "🥾 Adventure",\n    0.0, 1.0, 0.4, 0.1\n)\n\nphotography = st.sidebar.slider(\n    "📸 Photography",\n    0.0, 1.0, 0.8, 0.1\n)\n\nshopping = st.sidebar.slider(\n    "🛍️ Shopping",\n    0.0, 1.0, 0.2, 0.1\n)\n\nreligious = st.sidebar.slider(\n    "🛕 Religious",\n    0.0, 1.0, 0.1, 0.1\n)\n\nfamily = st.sidebar.slider(\n    "👨\u200d👩\u200d👧 Family",\n    0.0, 1.0, 0.4, 0.1\n)\n\nquery = st.text_area(\n    "💬 Describe your trip",\n    value=(\n        "I want a peaceful scenic trip with "\n        "beautiful places for photography."\n    ),\n    height=100\n)\n\npreferences = {\n    "nature": nature,\n    "history": history,\n    "culture": culture,\n    "adventure": adventure,\n    "photography": photography,\n    "shopping": shopping,\n    "religious": religious,\n    "family": family\n}\n\nrequest_payload = {\n    "destination": destination,\n    "query": query,\n    "days": int(days),\n    "top_n": int(top_n),\n    "preferences": preferences\n}\n\n# -------------------------------------------------\n# Generate results\n# -------------------------------------------------\n\ngenerate = st.button(\n    "✨ Generate My Trip",\n    type="primary",\n    use_container_width=True\n)\n\nif generate:\n\n    try:\n        with st.spinner("TravelMate is thinking... 🤖"):\n\n            recommendation_response = requests.post(\n                f"{API_BASE_URL}/recommend",\n                json=request_payload,\n                timeout=60\n            )\n\n            itinerary_response = requests.post(\n                f"{API_BASE_URL}/itinerary",\n                json=request_payload,\n                timeout=60\n            )\n\n        recommendation_response.raise_for_status()\n        itinerary_response.raise_for_status()\n\n        recommendations_data = (\n            recommendation_response\n            .json()\n        )\n\n        itinerary_data = (\n            itinerary_response\n            .json()\n        )\n\n        # -------------------------------------------------\n        # Recommendations\n        # -------------------------------------------------\n\n        st.header("🎯 Recommended Places")\n\n        recommendations = recommendations_data.get(\n            "recommendations",\n            []\n        )\n\n        if recommendations:\n\n            rec_df = pd.DataFrame(\n                recommendations\n            )\n\n            display_columns = [\n                "name",\n                "activity_type",\n                "rating",\n                "reviews",\n                "estimated_visit_minutes",\n                "api_score"\n            ]\n\n            available = [\n                col\n                for col in display_columns\n                if col in rec_df.columns\n            ]\n\n            st.dataframe(\n                rec_df[available],\n                use_container_width=True,\n                hide_index=True\n            )\n\n        else:\n            st.warning(\n                "No recommendations were returned."\n            )\n\n        # -------------------------------------------------\n        # Itinerary\n        # -------------------------------------------------\n\n        st.header("📅 Your Itinerary")\n\n        itinerary = itinerary_data.get(\n            "itinerary",\n            []\n        )\n\n        if not itinerary:\n            st.warning(\n                "No itinerary could be generated."\n            )\n        else:\n\n            itinerary_df = pd.DataFrame(\n                itinerary\n            )\n\n            for day in sorted(\n                itinerary_df["day"].unique()\n            ):\n\n                day_df = itinerary_df[\n                    itinerary_df["day"] == day\n                ].sort_values("stop")\n\n                st.subheader(\n                    f"📍 Day {int(day)}"\n                )\n\n                for _, row in day_df.iterrows():\n\n                    st.markdown(\n                        f"""\n                        **Stop {int(row[\'stop\'])}: {row[\'place\']}**  \n                        🕘 {row[\'arrival\']} → {row[\'departure\']}  \n                        🎯 Score: {row[\'score\']}  \n                        ⏱️ Visit: {int(row[\'visit_minutes\'])} min  \n                        🚗 Travel before stop: {row[\'travel_before_minutes\']} min\n                        """\n                    )\n\n                st.divider()\n\n        # -------------------------------------------------\n        # Map\n        # -------------------------------------------------\n\n        st.header("🗺️ Trip Map")\n\n        if itinerary:\n\n            valid_points = [\n                item\n                for item in itinerary\n                if item.get("latitude") is not None\n                and item.get("longitude") is not None\n            ]\n\n            if valid_points:\n\n                center_lat = sum(\n                    item["latitude"]\n                    for item in valid_points\n                ) / len(valid_points)\n\n                center_lon = sum(\n                    item["longitude"]\n                    for item in valid_points\n                ) / len(valid_points)\n\n                travel_map = folium.Map(\n                    location=[\n                        center_lat,\n                        center_lon\n                    ],\n                    zoom_start=13\n                )\n\n                for item in valid_points:\n\n                    folium.Marker(\n                        location=[\n                            item["latitude"],\n                            item["longitude"]\n                        ],\n                        popup=(\n                            f"Day {item[\'day\']} - "\n                            f"Stop {item[\'stop\']}<br>"\n                            f"{item[\'place\']}<br>"\n                            f"{item[\'arrival\']} - "\n                            f"{item[\'departure\']}"\n                        ),\n                        tooltip=(\n                            f"Day {item[\'day\']} - "\n                            f"{item[\'stop\']}: "\n                            f"{item[\'place\']}"\n                        )\n                    ).add_to(travel_map)\n\n                # Draw one route line for each day.\n                for day in sorted(\n                    set(\n                        item["day"]\n                        for item in valid_points\n                    )\n                ):\n\n                    day_points = [\n                        item\n                        for item in valid_points\n                        if item["day"] == day\n                    ]\n\n                    day_points = sorted(\n                        day_points,\n                        key=lambda x: x["stop"]\n                    )\n\n                    coordinates = [\n                        [\n                            item["latitude"],\n                            item["longitude"]\n                        ]\n                        for item in day_points\n                    ]\n\n                    if len(coordinates) >= 2:\n\n                        folium.PolyLine(\n                            coordinates,\n                            tooltip=f"Day {day} route"\n                        ).add_to(travel_map)\n\n                st_folium(\n                    travel_map,\n                    width=None,\n                    height=600\n                )\n\n    except requests.exceptions.ConnectionError:\n\n        st.error(\n            "❌ FastAPI is not running. "\n            "Start it first with: "\n            "`uvicorn app.main:app --reload`"\n        )\n\n    except requests.exceptions.Timeout:\n\n        st.error(\n            "⏳ The backend took too long to respond. "\n            "Please try again."\n        )\n\n    except requests.exceptions.RequestException as exc:\n\n        st.error(\n            f"❌ API request failed: {exc}"\n        )\n\n    except Exception as exc:\n\n        st.error(\n            f"❌ Unexpected error: {exc}"\n        )\n\nelse:\n\n    st.info(\n        "👈 Choose your preferences and click "\n        "**Generate My Trip**."\n    )\n\nst.caption(\n    "TravelMate AI • Recommendation + Itinerary + Map"\n)\n'

app_dir = Path("../app")
app_dir.mkdir(parents=True, exist_ok=True)

app_path = app_dir / "streamlit_app.py"

app_path.write_text(
    app_code,
    encoding="utf-8"
)

print(f"✅ Streamlit app created: {app_path}")


✅ Streamlit app created: ..\app\streamlit_app.py


In [4]:
print("Map integration ready ✅")


Map integration ready ✅
